## Compare all *_full.json per trait

Reads all `*_full.json` files from `test_outputs` and creates one wide comparison table:
- rows: traits
- columns: `run_name (metric)`
- metrics: `rmse`, `pearson_r`, `r2`, `mae`, `n_valid`

In [4]:
from pathlib import Path
import json
import re

import pandas as pd

metrics_dir = Path("/scratch3/plant-traits-v2/test_outputs")
json_files = sorted(metrics_dir.glob("*_full.json"))

if not json_files:
    raise FileNotFoundError(f"No '*_full.json' files found in: {metrics_dir}")

records = []
for file_path in json_files:
    run_name = file_path.name.replace(".test_metrics_full.json", "").replace("_full.json", "")

    payload = json.loads(file_path.read_text(encoding="utf-8"))
    trait_metrics = payload.get("trait_metrics", {})

    for trait_name, vals in trait_metrics.items():
        records.append(
            {
                "trait": trait_name,
                "run_name": run_name,
                "rmse": vals.get("rmse"),
                "pearson_r": vals.get("pearson_r"),
                "r2": vals.get("r2"),
                "mae": vals.get("mae"),
                "n_valid": vals.get("n_valid"),
            }
        )

if not records:
    raise ValueError("No trait metrics found in the JSON files.")

df = pd.DataFrame(records)
metric_cols = ["pearson_r"]

# Sort runs by trailing number, e.g. generous-star-98 ... glowing-sky-102
runs = df["run_name"].dropna().unique().tolist()

def run_sort_key(name: str) -> tuple[int, str]:
    m = re.search(r"(\d+)$", str(name))
    return (int(m.group(1)), str(name)) if m else (10**9, str(name))

run_order = sorted(runs, key=run_sort_key)
df["run_name"] = pd.Categorical(df["run_name"], categories=run_order, ordered=True)

wide = (
    df.pivot(index="trait", columns="run_name", values=metric_cols)
    .swaplevel(0, 1, axis=1)
)

# Keep columns ordered by run_order, then by metric_cols
ordered_cols = [(run, metric) for run in run_order for metric in metric_cols if (run, metric) in wide.columns]
wide = wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

wide.columns = [f"{run} ({metric})" for run, metric in wide.columns]

print(f"Loaded {len(json_files)} files from: {metrics_dir}")
print("Runs:", ", ".join(run_order))

wide

Loaded 5 files from: /scratch3/plant-traits-v2/test_outputs
Runs: generous-star-98, hearty-firebrand-99, stellar-bird-100, amber-shadow-101, glowing-sky-102


,generous-star-98 (pearson_r),hearty-firebrand-99 (pearson_r),stellar-bird-100 (pearson_r),amber-shadow-101 (pearson_r),glowing-sky-102 (pearson_r)
trait,,,,,
1080,0.387301,0.430719,0.375583,0.404208,0.422317
13,0.442487,0.484923,0.450051,0.436557,0.462541
138,0.395931,0.431401,0.398237,0.422331,0.405558
14,0.433908,0.462601,0.464476,0.469431,0.477006
144,0.631812,0.648841,0.642066,0.643574,0.644213
145,0.690790,0.713309,0.712073,0.717980,0.716319
146,0.437809,0.464298,0.453170,0.465194,0.478498
15,0.691588,0.703818,0.701419,0.687140,0.689862
163,0.776272,0.788442,0.792059,0.790945,0.790588
